<a href="https://colab.research.google.com/github/AdvayRoongta/audio-initial-dataset/blob/main/scripts/load_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Load the four source datasets

Pulls the first **100 rows** of each of the four public Hugging Face datasets
behind our conflict categories -- fast, for local iteration, not the final
dataset size -- and saves the audio + a browsable manifest.

| category | person | source dataset |
|---|---|---|
| `emotional_prosody` | Advay | [CREMA-D](https://huggingface.co/datasets/confit/cremad-parquet) |
| `non_speech_event` | Ankur | [ESC-50](https://huggingface.co/datasets/renumics/esc50) |
| `speaker_identity` | Danity | [RAVDESS](https://huggingface.co/datasets/xbgoose/ravdess) |
| `sarcasm_pacing` | Vishnu | [MUStARD](https://huggingface.co/datasets/macabdul9/SarcasmDetection_Mustard) |

Output:
```
data/raw/<category>/audio/<clip_id>.wav
data/raw/<category>/manifest.json   <- browse this to pick clips for your 20 examples
```


## Setup

In [ ]:
import os

# Land at the repo root regardless of how this notebook was opened:
#  - fresh Colab (via the badge above): only this notebook exists yet -> clone
#  - local Jupyter opened from inside scripts/ -> just cd up one level
#  - local Jupyter opened from the repo root -> already there, no-op
if os.path.exists("requirements.txt"):
    pass
elif os.path.exists("../requirements.txt"):
    %cd ..
else:
    !git clone -q https://github.com/AdvayRoongta/audio-initial-dataset.git repo
    %cd repo

!pip install -q -r requirements.txt


## Config: one dataset per category

In [ ]:
import json
from pathlib import Path

import soundfile as sf
from datasets import load_dataset

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "raw"
N_ROWS = 100  # keep small for fast iteration; bump later for the real dataset build

# One public, ungated Hugging Face dataset per conflict category / person.
# Each is loadable with plain `datasets.load_dataset(hf_id)` -- no
# trust_remote_code and no HF login required, so everyone can just run this.
CATEGORIES = {
    "emotional_prosody": {          # Advay: cheerful text vs. angry delivery, etc.
        "hf_id": "confit/cremad-parquet",   # CREMA-D: acted emotional speech
        "hf_split": "train",
        "primary_label_field": "emotion",
    },
    "non_speech_event": {            # Ankur: calm text vs. sirens/screams/etc. in audio
        "hf_id": "renumics/esc50",          # ESC-50: 50-class environmental sounds
        "hf_split": "train",
        "primary_label_field": "label",
    },
    "speaker_identity": {            # Danity: text claims one speaker, voice is another
        "hf_id": "xbgoose/ravdess",         # RAVDESS: acted speech, known actor gender
        "hf_split": "train",
        "primary_label_field": "gender",
    },
    "sarcasm_pacing": {              # Vishnu: sincere text vs. sarcastic delivery
        "hf_id": "macabdul9/SarcasmDetection_Mustard",  # MUStARD: TV-show sarcasm clips
        "hf_split": "test",              # this dataset only ships a "test" split
        "primary_label_field": "label",
    },
}


## Load + save each category

In [ ]:
def label_to_str(value, feature):
    """Some HF label columns are ints backed by a ClassLabel; others are plain strings."""
    if hasattr(feature, "int2str"):
        return feature.int2str(value)
    return str(value)


def load_category(name: str, cfg: dict, n_rows: int) -> None:
    print(f"\n=== {name} ({cfg['hf_id']}) ===")
    ds = load_dataset(cfg["hf_id"], split=cfg["hf_split"])
    if n_rows < len(ds):
        ds = ds.select(range(n_rows))

    out_dir = DATA_DIR / name
    audio_dir = out_dir / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)

    features = ds.features
    label_field = cfg["primary_label_field"]

    manifest = []
    for i, example in enumerate(ds):
        audio = example["audio"]  # {"array": np.ndarray, "sampling_rate": int, ...}
        clip_id = f"{name}_{i:03d}"
        audio_path = audio_dir / f"{clip_id}.wav"
        sf.write(audio_path, audio["array"], audio["sampling_rate"])

        primary_label = label_to_str(example[label_field], features[label_field])

        # Keep every other column (minus the raw audio array, which we already
        # wrote to disk) as free-form metadata so nothing from the source
        # dataset gets lost even if we didn't explicitly plan for a field.
        source_metadata = {k: v for k, v in example.items() if k != "audio"}

        manifest.append({
            "clip_id": clip_id,
            "audio_path": str(audio_path.relative_to(REPO_ROOT)),
            "duration_sec": round(len(audio["array"]) / audio["sampling_rate"], 3),
            "primary_label": primary_label,
            "source_metadata": source_metadata,
        })

    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
    print(f"saved {len(manifest)} clips -> {audio_dir}")
    print(f"wrote manifest  -> {manifest_path}")


for name, cfg in CATEGORIES.items():
    load_category(name, cfg, N_ROWS)


## Next

Browse `data/raw/<category>/manifest.json`, pick ~20 clips, and hand-write
conflict examples into `examples.json` (schema + a worked example per
category are in `README.md`). Then open `scripts/early_eval.ipynb`.